## Bronze Ingestion

Read the source CSV from ADLS Gen2 and save it as a Delta table. Data is kept in its original form, with ingestion metadata added.

## 1. Read source CSV

Read employees.csv from the ADLS Gen2 volume. Schema inference is disabled to keep the original values as strings.

In [0]:
raw_path = "/Volumes/employees_pipeline/bronze/source_files/employees.csv"

bronze_df = (
    spark.read
    .option("header", True)
    .option("inferSchema",False)
    .csv(raw_path)
)

## 2. Check source data

Check the row count, column types and sample records before writing to Bronze.

In [0]:
bronze_df.count()

In [0]:
bronze_df.printSchema()

In [0]:
bronze_df.show(10, truncate=False)

## 3. Add ingestion metadata

Add two columns:
- _ingested_at — ingestion timestamp
- _source_file — source file name

In [0]:
from pyspark.sql.functions import current_timestamp, lit

bronze_df = (
    bronze_df
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("employees.csv"))
)

In [0]:
bronze_df.select(
    "employee_id",
    "_ingested_at",
    "_source_file"
).show(5, truncate=False)

## 4. Save Bronze table

Write the data to a Delta table in Unity Catalog. No business validation or cleaning is done at this stage.

In [0]:


(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("employees_pipeline.bronze.employees_bronze")
)



## 5. Check Bronze table

Read the saved Delta table and check the row count.

In [0]:


bronze_table_df = spark.table(
    "employees_pipeline.bronze.employees_bronze"
)

print("BRONZE records:", bronze_table_df.count())



## Bronze Summary

- Source: employees.csv (ADLS Gen2)
- Source rows: 1000
- Bronze rows: 1000
- Table: employees_pipeline.bronze.employees_bronze
- Format: Delta
- Cleaning: Not applied